#### Padas 的数据组合函数:concat、merge、join

In [1]:
import pandas as pd

concat 连接

In [2]:
# Series 与 Series 连接
s1 = pd.Series(["A", "B"], index=[1, 2])
s2 = pd.Series(["D", "E"], index=[4, 5])
s3 = pd.Series(["G", "H"], index=[7, 8])
pd.concat([s1, s2, s3]) # 按行连接

1    A
2    B
4    D
5    E
7    G
8    H
dtype: str

In [3]:
# 按列链接
pd.concat([s1, s2, s3], axis=1)

,0,1,2
1,A,NaN,NaN
2,B,NaN,NaN
4,NaN,D,NaN
5,NaN,E,NaN
7,NaN,NaN,G
8,NaN,NaN,H


DataFrame与Series连接

In [4]:
df1 = pd.DataFrame(data={"a": [1, 2], "b": [4, 5]}, index=[1, 2])
s1 = pd.Series(data=[7, 10], index=[1, 2], name="a")

In [5]:
df1

,a,b
1,1,4
2,2,5


In [6]:
s1

1     7
2    10
Name: a, dtype: int64

In [7]:
#按行连接
df = pd.concat([df1, s1]) 
df.reset_index()

,index,a,b
0,1,1,4.0
1,2,2,5.0
2,1,7,NaN
3,2,10,NaN


In [8]:
#按列连接
pd.concat([df1, s1], axis=1) 

,a,b,a
1,1,4,7
2,2,5,10


DataFrame与DataFrame连接

In [9]:
df1 = pd.DataFrame(data=[{'a':1,'b':4},{'a':2, 'b':5}], index=[1, 2])
df2 = pd.DataFrame(data={"a": [7, 8], "b": [10, 11]}, index=[1, 2])

In [10]:
df1

,a,b
1,1,4
2,2,5


In [11]:
df2

,a,b
1,7,10
2,8,11


In [12]:
# 按列连接
pd.concat([df1, df2])

,a,b
1,1,4
2,2,5
1,7,10
2,8,11


In [13]:
# 按行连接
df2.index=[3,4]
pd.concat([df1, df2], axis = 1)

,a,b,a,b
1,1.0,4.0,NaN,NaN
2,2.0,5.0,NaN,NaN
3,NaN,NaN,7.0,10.0
4,NaN,NaN,8.0,11.0


In [14]:
# 无视索引直接“append”
pd.concat([df1, df2], ignore_index=True)

,a,b
0,1,4
1,2,5
2,7,10
3,8,11


In [15]:
# df的concat除了“append”功能外，还支持类似数据库表的连接查询

In [16]:
df1 = pd.DataFrame(data={"a": [1, 2], "b": [4, 5]}, index=[1, 2]) #连接之后只保留公共列
df2 = pd.DataFrame(data={"b": [7, 8], "c": [10, 11]}, index=[2, 3]) #连接之后保留所有
pd.concat([df1, df2], join='inner')

,b
1,4
2,5
2,7
3,8


In [17]:
pd.concat([df1, df2], join='outer')

,a,b,c
1,1.0,4,NaN
2,2.0,5,NaN
2,NaN,7,10.0
3,NaN,8,11.0


merge实现真正的“连接”操作

In [18]:
df1 = pd.DataFrame({"employee": ["Bob", "Jake", "Lisa", "Sue"], "group": ["Accounting","Engineering", "Engineering", "HR"]})
df2=pd.DataFrame({"employee":["Lisa","Bob","Jake","Sue"],"hire_date":[2004, 2008, 2012, 2014]})

In [19]:
df1

,employee,group
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR


In [20]:
df2

,employee,hire_date
0,Lisa,2004
1,Bob,2008
2,Jake,2012
3,Sue,2014


In [21]:
pd.merge(df1, df2)

,employee,group,hire_date
0,Bob,Accounting,2008
1,Jake,Engineering,2012
2,Lisa,Engineering,2004
3,Sue,HR,2014


In [22]:
# 连接过程可能出现1:n的情形，此时保留n行
df1 = pd.DataFrame({"employee": ["Bob", "Jake", "Lisa", "Sue"], "group": ["Accounting","Engineering", "Engineering", "HR"]})
df2 = pd.DataFrame({"group": ["Accounting", "Engineering", "HR"],"supervisor": ["Carly", "Guido", "Steve"]})

In [23]:
df1

,employee,group
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR


In [24]:
df2

,group,supervisor
0,Accounting,Carly
1,Engineering,Guido
2,HR,Steve


In [25]:
pd.merge(df1, df2)

,employee,group,supervisor
0,Bob,Accounting,Carly
1,Jake,Engineering,Guido
2,Lisa,Engineering,Guido
3,Sue,HR,Steve


连接的两张表可能是“多对多”，即公共列值均有重复的行

In [26]:
df1 = pd.DataFrame(
{"employee": ["Bob", "Jake", "Lisa", "Sue"], "group": ["Accounting",
"Engineering", "Engineering", "HR"]}
)
df2 = pd.DataFrame(
{
"group":["Accounting","Accounting","Engineering","Engineering",
"HR", "HR"],
"skills": ["math", "spreadsheets", "coding", "linux",
"spreadsheets", "organization"],
}
)

In [27]:
df1

,employee,group
0,Bob,Accounting
1,Jake,Engineering
2,Lisa,Engineering
3,Sue,HR


In [28]:
df2

,group,skills
0,Accounting,math
1,Accounting,spreadsheets
2,Engineering,coding
3,Engineering,linux
4,HR,spreadsheets
5,HR,organization


In [29]:
pd.merge(df1, df2)

,employee,group,skills
0,Bob,Accounting,math
1,Bob,Accounting,spreadsheets
2,Jake,Engineering,coding
3,Jake,Engineering,linux
4,Lisa,Engineering,coding
5,Lisa,Engineering,linux
6,Sue,HR,spreadsheets
7,Sue,HR,organization


可以显式地指定连接用的公共列

In [30]:
pd.merge(df1, df2, on='group')

,employee,group,skills
0,Bob,Accounting,math
1,Bob,Accounting,spreadsheets
2,Jake,Engineering,coding
3,Jake,Engineering,linux
4,Lisa,Engineering,coding
5,Lisa,Engineering,linux
6,Sue,HR,spreadsheets
7,Sue,HR,organization


如果join on所用的公共列列名不同，也可以用left_on、right_on指定

In [31]:
pd.merge(df1, df2, left_on='group',right_on='group')

,employee,group,skills
0,Bob,Accounting,math
1,Bob,Accounting,spreadsheets
2,Jake,Engineering,coding
3,Jake,Engineering,linux
4,Lisa,Engineering,coding
5,Lisa,Engineering,linux
6,Sue,HR,spreadsheets
7,Sue,HR,organization


left_index或right_index参数设置为真时，以索引作为公共列

In [32]:
df1.set_index('group', inplace=True)

In [33]:
df2.set_index('group', inplace=True)

In [34]:
pd.merge(df1,df2,left_index=True,right_index=True)

,employee,skills
group,,
Accounting,Bob,math
Accounting,Bob,spreadsheets
Engineering,Jake,coding
Engineering,Jake,linux
Engineering,Lisa,coding
Engineering,Lisa,linux
HR,Sue,spreadsheets
HR,Sue,organization


In [35]:
import pandas as pd

df1 = pd.DataFrame({'A': [1, 2, 3], 'B': [4, 5, 6]}, index=['x', 'y', 'z'])
df2 = pd.DataFrame({'C': [7, 8, 9], 'D': [10, 11, 12]}, index=['y', 'z', 'w'])

# 左连接（默认）
result = df1.join(df2)
print(result)

   A  B    C     D
x  1  4  NaN   NaN
y  2  5  7.0  10.0
z  3  6  8.0  11.0


dataframe实现了join方法，功能和merge类似，不指定 on 参数时，join 会使用左侧 DataFrame 的索引与右侧 DataFrame 的索引进行连接

In [36]:
df1 = pd.DataFrame({
'key': ['A', 'B', 'C'],
'value1': [1, 2, 3]
})

df2 = pd.DataFrame({
'key': ['B', 'C', 'D'],
'value2': [4, 5, 6]
})

#合并两个DataFrame时如果列名重复，可以通过suffix参数处理列名冲突
df1.join(df2,lsuffix='_left',rsuffix='_right')

,key_left,value1,key_right,value2
0,A,1,B,4
1,B,2,C,5
2,C,3,D,6


通过 how 参数可以指定不同的连接逻辑：

| `how` 参数 | 含义     | 结果包含哪些行                                               |
| :--------- | :------- | :----------------------------------------------------------- |
| `'left'`   | 左连接   | 保留左侧 DataFrame 的所有行，右侧匹配不到的填充为 `NaN`      |
| `'right'`  | 右连接   | 保留右侧 DataFrame 的所有行，左侧匹配不到的填充为 `NaN`      |
| `'outer'`  | 全外连接 | 保留左右两侧的所有行，匹配不到的填充为 `NaN`                 |
| `'inner'`  | 内连接   | 只保留两侧都能匹配上的行                                     |
| `'cross'`  | 交叉连接 | 生成两个 DataFrame 所有行的笛卡尔积（此时 `on` 参数会被忽略） |

In [37]:
df1 = pd.DataFrame({"name": ["Peter", "Paul", "Mary"], "food": ["fish",
"beans", "bread"]}, columns=["name", "food"])
df2=pd.DataFrame({"name":["Mary","Joseph"],"drink":["wine","beer"]},
columns=["name", "drink"])

In [38]:
pd.merge(df1, df2, how="left")

,name,food,drink
0,Peter,fish,NaN
1,Paul,beans,NaN
2,Mary,bread,wine


连接过程中左右表如果有同名列，合并后两列均保留，pandas为其准备了固定的后缀以便区分

In [39]:
df1=pd.DataFrame({"name":["Bob","Jake","Lisa","Sue"],"rank":[1,2,3, 4]})
df2=pd.DataFrame({"name":["Bob","Jake","Lisa","Sue"],"rank":[3,1,4, 2]})

In [40]:
print(pd.merge(df1, df2, on="name")) #不指定后缀名，默认为_x和_y

   name  rank_x  rank_y
0   Bob       1       3
1  Jake       2       1
2  Lisa       3       4
3   Sue       4       2


也可以手动指定后缀

In [41]:
print(pd.merge(df1, df2, on="name", suffixes=("_df1", "_df2")))

   name  rank_df1  rank_df2
0   Bob         1         3
1  Jake         2         1
2  Lisa         3         4
3   Sue         4         2
